In [1]:
import os
import sys

In [2]:
sys.path.append(os.path.abspath("../.."))

In [3]:
from datalemur_utils import extract_iframe_from_url

In [4]:
url = "https://datalemur.com/questions/laptop-mobile-viewership"
extract_iframe_from_url(url)

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, count, col

In [6]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/03 02:03:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
viewership = spark.read\
.option("delimiter", "\t")\
.option("header", "true")\
.option("inferSchema", "false")\
.schema("user_id int, device_type string, view_time string")\
.format("csv")\
.load("viewership.txt")

In [8]:
viewership.createOrReplaceTempView("viewership")

In [9]:
spark.sql(
    """
    select 
    count(case when device_type in ('phone', 'tablet') then view_time end) as mobile_views,
    count(case when device_type in ('laptop') then view_time end) as laptop_views
    from viewership
    """
).show(truncate=False)

[Stage 0:>                                                          (0 + 1) / 1]

+------------+------------+
|mobile_views|laptop_views|
+------------+------------+
|3           |2           |
+------------+------------+



In [10]:
viewership\
.select(
    count(when(col("device_type").isin(["phone", "tablet"]), "view_time")).alias("mobile_views"),
    count(when(col("device_type").isin(["laptop"]), "view_time")).alias("laptop_views")
)\
.show(truncate=False)

+------------+------------+
|mobile_views|laptop_views|
+------------+------------+
|3           |2           |
+------------+------------+



In [11]:
spark.stop()